# API с IGDB

## Контекст:
IGDB - это база данных игр, которая доступна по API.
**Почти тоже самое, что и IMDB, только для игр...**

Через нее монжо получать структурированные поля:
|поле|что это|зачем|
|-|-|-|
|id|id игры|связка записей|
|name|название|основа мэтчинга|
|slug|нормализованное имя|ссылки и мэтчинг|
|first_release_date|первая дата релиза|время и фильтры|
|platforms.name|платформы|разделять релизы|
|genres.name|жанры|сегментация|
|themes.name|темы|контекст|
|game_modes.name|режимы|тип опыта|
|game_type|тип игры|отделять dlc и ремастеры|
|involved_companies.company.name|компания|студия/издатель|
|rating|рейтинг IGDB|оценка|
|rating_count|число оценок|надежность рейтинга|
|aggregated_rating|рейтинг критиков|внешняя оценка|
|aggregated_rating_count|число критических оценок|надежность aggregated|
|total_rating|общий рейтинг|сводная оценка|
|total_rating_count|число оценок total|контекст|
|hypes|интерес до релиза|хайп|
|external_games.uid|внешний id|склейка с платформами|
|external_games.external_game_source|источник внешнего id|откуда матчим|

Доступ к IGDB возможен только по токену, который мы успешно сгенерировали с помощью Twitch Developer. Подробнее о том, как это сделать можно узнать в [документации IGDB](https://api-docs.igdb.com/#getting-started)

В следующих этапах проекта, мы объединим данные с API со скреппингом (либо используем иначе).

## Начнем с настройки API
Чтобы наши прелестные ключики не улетели в свободное использование на github (так как репозиторий публичен), подтянем их из окружения, до этого закинув их в терминал

In [2]:
import os

In [ ]:
os.environ['twitch_client_secret'] = "67sixseven"
os.environ['twitch_client_id'] = "67sixseven"

#пж замените на данные в тгшке, но не пуште с секретками

In [5]:
cid = os.environ['twitch_client_id']
secret = os.environ['twitch_client_secret']

In [6]:
# а теперь получим bearer token для доступа к API IGDB
import requests
import json
resp = requests.post('https://id.twitch.tv/oauth2/token', data={
    'client_id': cid,
    'client_secret': secret,
    'grant_type': 'client_credentials'
})

data = resp.json()
# data

In [8]:
# data['access_token']

{'access_token': 'сиксевен676767',
 'expires_in': 5283769,
 'token_type': 'bearer'}

In [ ]:
os.environ['access_token'] = 'сиксевен676767'
# os.getenv('access_token')

В С Ë. Мы получили токен для работы с API. Дальше будем работать напрямую с докой и искать данные

Так как по условию нам нужно использовать не менее 5 эндпойнтов — было принято решение написать функцию

### Всю последующую работу я делаю опираясь на Seminar 8 - API

Далее сделаем request. В основном из методов в документации только post. Обращаемся к `https://api.igdb.com/v4`. 

Еще я добавил логгирование чтобы отслеживать важные моменты и отлавливать места с проблемаии

In [11]:
import logging 
logging.basicConfig(level=logging.INFO)

In [12]:
token = os.getenv('access_token')
cid

'po0syvw9g7dh0et8fdhse6y12l4gmu'

In [13]:
# этот url нам понадобится для запросов к API IGDB
url = 'https://api.igdb.com/v4/games'

# в headers идентифицируем себя и указываем, что хотим получить json
headers = {
    'Client-ID': cid,
    'Authorization': f"Bearer {os.getenv('access_token')}",
    'Accept': 'application/json'
}

# а теперь делаем запрос к API IGDB, чтобы получить 10 игр с их рейтингами
body = 'fields name, rating; limit 10;'

logging.info("Отправляем запрос к API IGDB...")
#добавил еще таймаут, чтобы не зависало на запросе
games10 = requests.post(url, headers=headers, data=body, timeout=10)
logging.info(f'Ответ от IGDB {games10.status_code}')

INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200


УРААААА!!! Ответ 200, у нас есть data. Давайте посмотрим как там внутри

In [15]:
games10.json()

[{'id': 350392, 'name': 'Rival Species'},
 {'id': 20243, 'name': 'Wipeout 2', 'rating': 75.0},
 {'id': 207571, 'name': 'A Very Corporate Escape Room'},
 {'id': 339266, 'name': 'Power Guy World'},
 {'id': 63844, 'name': 'Ace wo Nerae!', 'rating': 52.90462943179914},
 {'id': 371149, 'name': 'The Deal'},
 {'id': 330684, 'name': 'Nightmare Kart: The Old Karts'},
 {'id': 371776, 'name': 'Born for the Light: Wavelength Radiant Collapse'},
 {'id': 129477, 'name': 'Chomp Chomp'},
 {'id': 179243,
  'name': 'Satella-Q: Mou Sugu Haru desu ne - Chotto Sate-Q Shimasen ka?'}]

Обратим внимание, что рейтинг есть не везде. Покопавшись чуть глубже в доке узнал, что поле с рейтингом является опциональном. Вообще, покопавшись в метриках можно заметить, что там целый набор метрик: rating, rating_count, aggregated_rating, aggregated_rating_count, total_rating и total_rating_count. 

В таком случае имеет место быть заселектить разные поля, а самое главное поставить фильтры... Я не до конца понимаю по какому принципу сортируется сейчас, поэтому лучше добавить фильтры и сортировку...

In [16]:
# сделаем функцию чтобы постоянно не писать эти заголовки и url
def igdb_request(body, timeout=10):
    # этот url нам понадобится для запросов к API IGDB
    url = 'https://api.igdb.com/v4/games'

    # в headers идентифицируем себя и указываем, что хотим получить json
    headers = {
        'Client-ID': cid,
        'Authorization': f"Bearer {os.getenv('access_token')}",
        'Accept': 'application/json'
    }

    logging.info("Отправляем запрос к API IGDB...")
    data_req = requests.post(url, headers=headers, data=body, timeout=timeout)
    logging.info(f'Ответ от IGDB {data_req.status_code}')

    return data_req.json()

### СЕЛЕКТИМ ВСЕ
Селектим все данные, где есть рейтинг и количество оценок >20


In [23]:
body='''
fields
    id,
    name,
    slug,
    first_release_date,
    platforms.name,
    genres.name,
    themes.name,
    game_modes.name,
    game_type,
    involved_companies.company.name,
    rating,
    rating_count,
    aggregated_rating,
    aggregated_rating_count,
    total_rating,
    total_rating_count,
    hypes,
    external_games.uid,
    external_games.external_game_source;
where rating!=null & rating_count>20;
sort rating desc;
limit 500;
'''
data_new = igdb_request(body)

INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200


In [20]:
import pandas as pd 
df = pd.DataFrame(data_new)
df 

,id,external_games,first_release_date,game_modes,genres,involved_companies,name,platforms,rating,rating_count,slug,themes,total_rating,total_rating_count,game_type,aggregated_rating,aggregated_rating_count,hypes
0,65755,"[{'id': 1896086, 'uid': '32308', 'external_gam...",1311897600,"[{'id': 1, 'name': 'Single player'}]","[{'id': 2, 'name': 'Point-and-click'}, {'id': ...","[{'id': 55465, 'company': {'id': 13608, 'name'...",Trailer Park King,"[{'id': 12, 'name': 'Xbox 360'}]",100.000000,47,trailer-park-king,"[{'id': 27, 'name': 'Comedy'}]",100.000000,47,0,NaN,NaN,NaN
1,112659,"[{'id': 1929804, 'uid': 'B083T65GQX', 'externa...",1543276800,"[{'id': 1, 'name': 'Single player'}]","[{'id': 25, 'name': 'Hack and slash/Beat 'em u...","[{'id': 107496, 'company': {'id': 14055, 'name...",Batman: Arkham Collection,"[{'id': 48, 'name': 'PlayStation 4'}, {'id': 6...",99.584603,24,batman-arkham-collection,"[{'id': 1, 'name': 'Action'}, {'id': 20, 'name...",99.584603,24,3,NaN,NaN,NaN
2,20196,"[{'id': 1933251, 'uid': 'B00CE5K2M0', 'externa...",1373328000,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 24, 'nam...","[{'id': 37443, 'company': {'id': 129, 'name': ...",Metal Gear Solid: The Legacy Collection,"[{'id': 9, 'name': 'PlayStation 3'}]",99.513151,43,metal-gear-solid-the-legacy-collection,"[{'id': 1, 'name': 'Action'}, {'id': 23, 'name...",93.256575,45,3,87.0,2.0,NaN
3,47304,"[{'id': 1871861, 'uid': '10867', 'external_gam...",1103673600,NaN,"[{'id': 10, 'name': 'Racing'}]",NaN,American Chopper,"[{'id': 11, 'name': 'Xbox'}, {'id': 6, 'name':...",99.457627,49,american-chopper,"[{'id': 1, 'name': 'Action'}]",99.457627,49,0,NaN,NaN,NaN
4,45131,"[{'id': 1933044, 'uid': 'B00CY92XU0', 'externa...",1379376000,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 104254, 'company': {'id': 139, 'name':...",Grand Theft Auto V: Special Edition,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 12...",99.310093,81,grand-theft-auto-v-special-edition,"[{'id': 1, 'name': 'Action'}, {'id': 27, 'name...",99.310093,81,0,NaN,NaN,NaN
5,136482,"[{'id': 1993429, 'uid': '1521500257', 'externa...",1702080000,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 232886, 'company': {'id': 27350, 'name...",Undertale Yellow,"[{'id': 6, 'name': 'PC (Microsoft Windows)'}]",99.221116,24,undertale-yellow,"[{'id': 17, 'name': 'Fantasy'}, {'id': 27, 'na...",99.221116,24,0,NaN,NaN,NaN
6,135478,"[{'id': 1883312, 'uid': '518311', 'external_ga...",1684195200,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 32, 'nam...","[{'id': 102930, 'company': {'id': 26662, 'name...",Hrot,"[{'id': 6, 'name': 'PC (Microsoft Windows)'}]",99.074816,27,hrot,"[{'id': 1, 'name': 'Action'}, {'id': 19, 'name...",93.037408,28,0,87.0,1.0,NaN
7,143761,"[{'id': 1998461, 'uid': '1044890345', 'externa...",1429660800,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...",NaN,Pokémon Infinite Fusion,"[{'id': 6, 'name': 'PC (Microsoft Windows)'}]",98.470805,21,pokemon-infinite-fusion,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",98.470805,21,0,NaN,NaN,NaN
8,165192,"[{'id': 2169207, 'uid': 'B09L22J1S9', 'externa...",1636588800,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 145759, 'company': {'id': 126, 'name':...",The Elder Scrolls V: Skyrim - Anniversary Edition,"[{'id': 169, 'name': 'Xbox Series X|S'}, {'id'...",98.300401,69,the-elder-scrolls-v-skyrim-anniversary-edition,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",91.150200,71,10,84.0,2.0,NaN
9,11631,"[{'id': 104353, 'uid': '57646', 'external_game...",1505347200,"[{'id': 1, 'name': 'Single player'}]","[{'id': 2, 'name': 'Point-and-click'}, {'id': ...","[{'id': 86598, 'company': {'id': 18629, 'name'...",Hiveswap: Act 1,"[{'id': 3, 'name': 'Linux'}, {'id': 6, 'name':...",98.112915,22,hiveswap-act-1,NaN,86.556457,23

Итак... Мало того, что многие ячейки остались внутри json-ами(это поправимо на постобработке eda), так мы еще и увидели, что заселектилось всего 10 ячеек. Опытным путем и игрой с лимитами, выяснялось — что нормально селектятся 500- строк...

Изучая проблему, я осознал, что мы не можем полностью заселектить все. Но можем селектить БАТЧАМИ... О том как это делается я прочитал в [статье на proglib](https://proglib.io/p/paketnyy-api-obedinenie-zaprosov-s-pomoshchyu-asyncio-i-batch-api-2023-03-23).

Еще я добавил TQDM, чтобы мы могли отслеживать сколько времени осталось и на каком мы этапе...
Берем данные от самых больших по количеству отзывов, до самых маленьких



In [25]:
# !pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


In [26]:
from tqdm import tqdm
import time 

# Парсим ИГРЫ

In [ ]:
all_data=[]

for offset in tqdm(range(0,15000,500)):
    body=f'''
    fields
        id,
        name,
        slug,
        first_release_date,
        platforms.name,
        genres.name,
        themes.name,
        game_modes.name,
        game_type,
        involved_companies.company.name,
        rating,
        rating_count,
        aggregated_rating,
        aggregated_rating_count,
        total_rating,
        total_rating_count,
        hypes,
        external_games.uid,
        external_games.external_game_source;
    where rating_count>0;
    sort rating_count desc;
    limit 500;
    offset {offset};
    '''
    
    batch=igdb_request(body)
    
    if not batch:
        logging.info(f'на offset={offset} данные закончились')
        break
    
    all_data.extend(batch)
    time.sleep(0.8) # IGDB позволяет делать 4 запроса в секунду, так что добавляем паузу для успокоения

logging.info(f'Всего игр получено: {len(all_data)}')

  0%|          | 0/30 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
  3%|▎         | 1/30 [00:02<01:26,  2.97s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
  7%|▋         | 2/30 [00:05<01:16,  2.73s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 10%|█         | 3/30 [00:08<01:14,  2.77s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 13%|█▎        | 4/30 [00:11<01:12,  2.78s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 17%|█▋        | 5/30 [00:13<01:05,  2.63s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 20%|██        | 6/30 [00:16<01:02,  2.58s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 23%|██▎       | 7/30 [00:18<00:55,  2.42s/it]INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200
 27%|██▋       | 8/30 [00:20<00:51,  2.36s/it]INFO:root:Отправляем запрос к API 

In [6]:
df = pd.DataFrame(all_data)
df

NameError: name 'all_data' is not defined

In [5]:
df

NameError: name 'df' is not defined

Ура! Мы получили +- вменяемые данные. Теперь можем их обрабатывать далее...


На всякий ПОЖАРНЫЙ сохраним все в pickle, чтобы потом каждый раз не парсить заново

In [8]:
!pip install pyarrow
import pandas as pd 

In [9]:
df = pd.read_pickle("games.pkl")
df

,id,aggregated_rating,aggregated_rating_count,external_games,first_release_date,game_modes,genres,involved_companies,name,platforms,rating,rating_count,slug,themes,total_rating,total_rating_count,game_type,hypes
0,1020,88.137931,27.0,"[{'id': 1935351, 'uid': 'B00NOP3QF4', 'externa...",1.379376e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 20018, 'company': {'id': 139, 'name': ...",Grand Theft Auto V,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 48...",89.611826,5609,grand-theft-auto-v,"[{'id': 1, 'name': 'Action'}, {'id': 27, 'name...",88.874879,5636,0,NaN
1,1942,91.730769,26.0,"[{'id': 1935171, 'uid': 'B00T3SPV36', 'externa...",1.431994e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 17436, 'company': {'id': 50, 'name': '...",The Witcher 3: Wild Hunt,"[{'id': 169, 'name': 'Xbox Series X|S'}, {'id'...",93.791526,5165,the-witcher-3-wild-hunt,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",92.761147,5191,0,179.0
2,72,92.444444,9.0,"[{'id': 189642, 'uid': 'UCnj3zKNoYGZLWzFAo-bB3...",1.303085e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 8, 'name': 'Platform'}, {'id': 9, 'nam...","[{'id': 265481, 'company': {'id': 56, 'name': ...",Portal 2,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 3,...",91.287710,4255,portal-2,"[{'id': 1, 'name': 'Action'}, {'id': 18, 'name...",91.866077,4264,0,NaN
3,472,79.916667,10.0,"[{'id': 1933029, 'uid': 'B0050SYNYQ', 'externa...",1.320883e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 3719, 'company': {'id': 126, 'name': '...",The Elder Scrolls V: Skyrim,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 6,...",87.694031,4148,the-elder-scrolls-v-skyrim,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",83.805349,4158,0,NaN
4,732,93.142857,6.0,"[{'id': 1933194, 'uid': 'B0183O87HM', 'externa...",1.098749e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 3,...","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 55979, 'company': {'id': 365, 'name': ...",Grand Theft Auto: San Andreas,"[{'id': 11, 'name': 'Xbox'}, {'id': 9, 'name':...",90.155054,3844,grand-theft-auto-san-andreas,"[{'id': 1, 'name': 'Action'}, {'id': 23, 'name...",91.648956,3850,0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,1395,20.000000,1.0,"[{'id': 250499, 'uid': '26587', 'external_game...",1.269907e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 13, 'name': 'Simulator'}, {'id': 14, '...","[{'id': 20941, 'company': {'id': 858, 'name': ...",Dead or Alive Paradise,"[{'id': 38, 'name': 'PlayStation Portable'}]",33.249822,6,dead-or-alive-paradise,"[{'id': 1, 'name': 'Action'}, {'id': 27, 'name...",26.624911,7,8,NaN
14996,1208,NaN,NaN,"[{'id': 137585, 'uid': '21444', 'external_game...",1.196899e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 12, 'name': 'Role-playing (RPG)'}]","[{'id': 2912, 'company': {'id': 828, 'name': '...",Tales of Innocence,"[{'id': 46, 'name': 'PlayStation Vita'}, {'id'...",69.959030,6,tales-of-innocence,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",69.959030,6,0,NaN
14997,1054,50.000000,2.0,"[{'id': 1922882, 'uid': '33494', 'external_gam...",1.325376e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 11, 'name': 'Real Time Strategy (RTS)'...","[{'id': 2508, 'company': {'id': 742, 'name': '...",Oil Rush,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 3,...",58.476051,6,oil-rush,"[{'id': 18, 'name': 'Science fiction'}]",54.238026,8,0,NaN
14998,1024,NaN,NaN,"[{'id': 1922268, 'uid': '29416', 'external_gam...",1.229299e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 13, 'name': 'Simulator'}, {'id': 15, '...","[{'id': 7881, 'company': {'id': 737, 'name': '...",Hacker Evolution: Untold,"[{'id': 3, 'name': 'Linux'}, {'id': 6, 'name':...",52.746887,6,hacker-evolution-untold,"[{'id': 32, 'name': 'Non-fiction'}]",52.746887,6,0,NaN
